<a href="https://colab.research.google.com/github/Lyanan85/In-GameSeries/blob/venv/MATERIAL_Python__Normalizando_arquivos_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nesse material a gente vai lembrar das formas de normalização de um arquivo JSON transformando esse arquivo em um DataFrame manipulável.

> ***Nota:*** *Todos os dados utilizados nos exemplos desse notebook são fictícios.*

Quando trabalhamos com arquivos JSON, é importante saber como os dados são dispostos em arquivos como esse:
- Arquivos JSON armazenam dados em pares chave-valor (formato de dicionário) ou listas.
- No `arquivo_1.json`, temos uma lista em que cada elemento é um dicionário que se refere a uma linha do conjunto de dados, mostrando as colunas e os dados.

Vamos importar esse arquivo e transformá-lo em um dataframe!

Utilizaremos a biblioteca [`pandas`](https://pandas.pydata.org/docs/) para ler o arquivo JSON.

In [ ]:
import pandas as pd

Lemos o arquivo JSON com a funcionalidade [`pd.read_json`](https://pandas.pydata.org/docs/reference/api/pandas.read_json.html)

In [ ]:
df = pd.read_json('arquivo_1.json')
df

,id_vendedor,id_cliente,produto,quantidade,valor_unitario,valor_total
0,101,1001,Notebook,1,3500,3500
1,102,1002,Smartphone,2,1200,2400
2,103,1003,Monitor,3,800,2400


Então conseguimos ter o resultado final.

Arquivos JSON, dependendo de como forem criados, podem ter informações aninhadas em níveis. Essa estrutura pode ser observada no `arquivo_2.json`.
- No `arquivo_2` temos a situação de um JSON aninhado com dois níveis de chaves.
- A função `pd.read_json` não vai transformar corretamente esse arquivo em um dataframe por que essa função só considera o primeiro nível de uma arquivo JSON.

In [ ]:
df = pd.read_json('arquivo_2.json')
df

,id_vendedor,id_cliente,detalhes_compra
0,101,1001,"{'produto': 'Notebook', 'quantidade': 1, 'valo..."
1,102,1002,"{'produto': 'Smartphone', 'quantidade': 2, 'va..."
2,103,1003,"{'produto': 'Monitor', 'quantidade': 3, 'valor..."


Para ter o dataframe ajustado corretamente, devemos aplicar a normalização.

Para aplicar a normalização, usamos a função [`pd.json_normalize`](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html) do pandas.

Mas antes de ir direto para a aplicação dela, aqui vem uma **dica**:
- Aplicar a normalização nesses casos de aninhamento pode ser mais trabalhoso se usarmos um dataframe, pois essa função recebe como argumento uma lista de dicionários, ou um dicionário, não um DataFrame Pandas.
- Se quisermos muito usar um dataframe, teríamos que passar a coluna onde está acontecendo o problema e ainda concatenar a coluna no dataframe original.

Então, seguiremos o modo mais tranquilo, que é fazer a leitura desse arquivo JSON e armazená-lo como uma lista Python.

Usamos a biblioteca [`json`](https://docs.python.org/3/library/json.html#) para manipular arquivos JSON.

Para ler o arquivo JSON, usamos a função [`json.load`](https://docs.python.org/3/library/json.html#json.load)

In [ ]:
import json

In [ ]:
with open('arquivo_2.json', 'r') as arquivo:
    dados = json.load(arquivo)
dados

[{'id_vendedor': 101,
  'id_cliente': 1001,
  'detalhes_compra': {'produto': 'Notebook',
   'quantidade': 1,
   'valor_unitario': 3500.0,
   'valor_total': 3500.0}},
 {'id_vendedor': 102,
  'id_cliente': 1002,
  'detalhes_compra': {'produto': 'Smartphone',
   'quantidade': 2,
   'valor_unitario': 1200.0,
   'valor_total': 2400.0}},
 {'id_vendedor': 103,
  'id_cliente': 1003,
  'detalhes_compra': {'produto': 'Monitor',
   'quantidade': 3,
   'valor_unitario': 800.0,
   'valor_total': 2400.0}}]

Para transformar isso em um DataFrame, basta enviar `dados` à função [`json_normalize`](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html).

In [ ]:
df = pd.json_normalize(dados)
df

,id_vendedor,id_cliente,detalhes_compra.produto,detalhes_compra.quantidade,detalhes_compra.valor_unitario,detalhes_compra.valor_total
0,101,1001,Notebook,1,3500.0,3500.0
1,102,1002,Smartphone,2,1200.0,2400.0
2,103,1003,Monitor,3,800.0,2400.0


As colunas aninhadas em `detalhes_compra` recebem o prefixo do nome da coluna `detalhes_compra` seguido de um `.`. Isso é o padrão da função, mas pode ser alterado através do parâmetro `sep`, conforme informado na documentação. Exemplo:

```python
df = pd.json_normalize(dados, sep='_')
df
```

Na **saída**, percebemos que o separador das colunas novas é o `'_'`:

|    |   id_vendedor |   id_cliente | detalhes_compra_produto   |   detalhes_compra_quantidade |   detalhes_compra_valor_unitario |   detalhes_compra_valor_total |
|---:|--------------:|-------------:|:--------------------------|-----------------------------:|---------------------------------:|------------------------------:|
|  0 |           101 |         1001 | Notebook                  |                            1 |                             3500 |                          3500 |
|  1 |           102 |         1002 | Smartphone                |                            2 |                             1200 |                          2400 |
|  2 |           103 |         1003 | Monitor                   |                            3 |

Um último detalhe sobre a normalização de arquivos JSON é o comportamento da função `pd.json_normalize` quando se depara com uma estrutura que contém listas, como a mostrada abaixo:

In [ ]:
dados = [
            {
                "id_vendedor": 101,
                "id_cliente": 1001,
                "compras": [
                    {"produto": "Notebook", "quantidade": 1, "valor_unitario": 3500.00, "valor_total": 3500.00},
                    {"produto": "Mouse", "quantidade": 2, "valor_unitario": 50.00, "valor_total": 100.00}
                ]
            },
            {
                "id_vendedor": 102,
                "id_cliente": 1002,
                "compras": [
                    {"produto": "Smartphone", "quantidade": 1, "valor_unitario": 1200.00, "valor_total": 1200.00},
                    {"produto": "Fone de Ouvido", "quantidade": 1, "valor_unitario": 200.00, "valor_total": 200.00}
                ]
            }
        ]

Se tentarmos transformar esses dados diretamente em um DataFrame:

In [ ]:
df = pd.json_normalize(dados)
df

,id_vendedor,id_cliente,compras
0,101,1001,"[{'produto': 'Notebook', 'quantidade': 1, 'val..."
1,102,1002,"[{'produto': 'Smartphone', 'quantidade': 1, 'v..."


O resultado não incluirá a expansão da lista em `compras`. Isso acontece porque `pd.json_normalize` expande somente estruturas do tipo dicionário. Quando encontra uma lista, ele não consegue continuar a transformação.

Para resolver esse problema, podemos usar o parâmetro `record_path`, especificando o nome da coluna que contém a lista a ser expandida.

**Atenção:** Ao usar apenas o parâmetro `record_path`, podemos perder as colunas de nível superior, como `id_vendedor` e `id_cliente`. Vamos verificar esse comportamento:

In [ ]:
df = pd.json_normalize(dados, record_path='compras')
df

,produto,quantidade,valor_unitario,valor_total
0,Notebook,1,3500.0,3500.0
1,Mouse,2,50.0,100.0
2,Smartphone,1,1200.0,1200.0
3,Fone de Ouvido,1,200.0,200.0


As colunas `id_vendedor` e `id_cliente` desaparecem! Para preservar essas informações, adicionamos o parâmetro `meta`, que define os campos de nível superior a serem mantidos no DataFrame:

In [ ]:
df = pd.json_normalize(dados, record_path='compras', meta = ['id_vendedor', 'id_cliente'])
df

,produto,quantidade,valor_unitario,valor_total,id_vendedor,id_cliente
0,Notebook,1,3500.0,3500.0,101,1001
1,Mouse,2,50.0,100.0,101,1001
2,Smartphone,1,1200.0,1200.0,102,1002
3,Fone de Ouvido,1,200.0,200.0,102,1002


E isso é tudo que você precisa saber para importar arquivos JSON, normalizar suas estruturas e utilizá-los em seus projetos de Data Science.

Agora é sua vez de praticar o conhecimento adquirido! Realize os exercícios e sinta-se à vontade para compartilhar o que aprendeu. Boas práticas!

In [2]:
import pandas as pd

In [4]:
df=pd.read_json("/content/questao_1.json")
df

,id_pedido,id_cliente,nome_cliente,id_produto,nome_produto,categoria,quantidade,preco_unitario,preco_total,data_pedido,status
0,1,101,Alice Silva,201,Camiseta Básica,Roupas,2,50,100,2025-01-01,Entregue
1,2,102,Bruno Souza,202,Tênis Esportivo,Calçados,1,200,200,2025-01-02,Pendente
2,3,103,Carla Pereira,203,Relógio Digital,Acessórios,1,150,150,2025-01-03,Entregue
3,4,104,Daniel Oliveira,204,Calça Jeans,Roupas,2,120,240,2025-01-04,Cancelado
4,15,115,Olivia Rocha,215,Tênis Casual,Calçados,1,220,220,2025-01-15,Entregue


In [5]:
df_1=pd.read_json("/content/questao_2_1.json")
df_1

,id_pedido,id_cliente,nome_cliente,id_produto,nome_produto,categorias,quantidade,preco_unitario,preco_total,data_pedido,status
0,10,110,João Vieira,210,Smartphone,"[Eletrônicos, Comunicação, Tecnologia]",1,2500,2500,2025-01-10,Pendente
1,11,111,Karen Martins,211,Mouse Gamer,"[Eletrônicos, Acessórios, Games]",1,150,150,2025-01-11,Entregue
2,12,112,Lucas Mendes,212,Bicicleta,"[Esportes, Lazer, Transporte]",1,1200,1200,2025-01-12,Cancelado
3,13,113,Mariana Lopes,213,Óculos de Sol,"[Acessórios, Moda, Proteção]",1,300,300,2025-01-13,Entregue
4,14,114,Natan Santos,214,Tablet,"[Eletrônicos, Tecnologia, Educação]",1,1500,1500,2025-01-14,Pendente


In [7]:
df_2=pd.read_json("/content/questao_2_2.json")
df_2

,id_pedido,id_cliente,nome_cliente,id_produto,nome_produto,categoria,quantidade,preco_unitario,preco_total,data_pedido,status,endereco
0,5,105,Eduarda Costa,205,Mochila Escolar,Acessórios,1,180,180,2025-01-05,Entregue,"{Rua das Flores, 120, Centro, São Paulo, SP, 0..."
1,6,106,Fernando Lima,206,Notebook,Eletrônicos,1,3000,3000,2025-01-06,Pendente,"{Avenida Paulista, 1578, Bela Vista, São Paulo..."
2,7,107,Gabriela Souza,207,Fone de Ouvido,Eletrônicos,1,250,250,2025-01-07,Entregue,"{Rua do Comércio, 45, Centro, Rio de Janeiro, ..."
3,8,108,Hugo Almeida,208,Jaqueta de Couro,Roupas,1,350,350,2025-01-08,Entregue,"{Rua das Palmeiras, 234, Botafogo, Rio de Jane..."
4,9,109,Isabela Nunes,209,Caderno Universitário,Papelaria,5,20,100,2025-01-09,Entregue,"{Rua da Paz, 789, Centro, Curitiba, PR, 80010-..."


In [8]:
import json

In [9]:
df_3=pd.read_json("/content/questao_3.json")
df_3

,id_pedido,id_cliente,nome_cliente,data_pedido,status,detalhes_compra
0,16,116,Paulo Barros,2025-01-16,Pendente,"{'id_produto': 216, 'nome_produto': 'Câmera Di..."
1,17,117,Quezia Fernandes,2025-01-17,Entregue,"{'id_produto': 217, 'nome_produto': 'Bolsa de ..."
2,18,118,Rafael Cardoso,2025-01-18,Cancelado,"{'id_produto': 218, 'nome_produto': 'Livro de ..."
3,19,119,Sofia Ribeiro,2025-01-19,Entregue,"{'id_produto': 219, 'nome_produto': 'Perfume',..."
4,20,120,Tiago Moraes,2025-01-20,Entregue,"{'id_produto': 220, 'nome_produto': 'Guarda-Ch..."


In [10]:
with open("/content/questao_3.json","r") as e:
  dados=json.load(e)

In [12]:
df=pd.json_normalize(dados)
df

,id_pedido,id_cliente,nome_cliente,data_pedido,status,detalhes_compra.id_produto,detalhes_compra.nome_produto,detalhes_compra.categoria,detalhes_compra.quantidade,detalhes_compra.preco_unitario,detalhes_compra.preco_total
0,16,116,Paulo Barros,2025-01-16,Pendente,216,Câmera Digital,Eletrônicos,1,2000.0,2000.0
1,17,117,Quezia Fernandes,2025-01-17,Entregue,217,Bolsa de Couro,Acessórios,1,400.0,400.0
2,18,118,Rafael Cardoso,2025-01-18,Cancelado,218,Livro de Ficção,Livros,2,35.0,70.0
3,19,119,Sofia Ribeiro,2025-01-19,Entregue,219,Perfume,Beleza,1,180.0,180.0
4,20,120,Tiago Moraes,2025-01-20,Entregue,220,Guarda-Chuva,Acessórios,1,50.0,50.0


In [13]:
with open("/content/questao_4.json","r") as e:
  dados=json.load(e)

In [14]:
df=pd.json_normalize(dados)
df

,id_pedido,data_pedido,status,cliente.id_cliente,cliente.nome_cliente,detalhes_compra.produto.id_produto,detalhes_compra.produto.nome_produto,detalhes_compra.produto.categoria,detalhes_compra.quantidade,detalhes_compra.preco_unitario,detalhes_compra.preco_total
0,21,2025-01-21,Entregue,121,Ursula Castro,221,Sandália Feminina,Calçados,1,90.0,90.0
1,22,2025-01-22,Pendente,122,Victor Andrade,222,Chuteira,Esportes,1,230.0,230.0
2,23,2025-01-23,Entregue,123,Wagner Neves,223,Carregador Portátil,Eletrônicos,1,100.0,100.0
3,24,2025-01-24,Entregue,124,Xavier Almeida,224,Bola de Futebol,Esportes,1,120.0,120.0
4,25,2025-01-25,Cancelado,125,Yasmin Dias,225,Maquiagem Completa,Beleza,1,250.0,250.0


In [15]:
with open("/content/questao_5.json","r") as e:
  dados=json.load(e)

In [16]:
dados

[{'id_pedido': 26,
  'cliente': {'id_cliente': 126, 'nome_cliente': 'Zeca Costa'},
  'compra': {'id_produto': 226,
   'nome_produto': 'Cadeira de Escritório',
   'categoria': 'Móveis',
   'quantidade': 1,
   'preco_unitario': 550.0,
   'preco_total': 550.0},
  'data_pedido': '2025-01-26',
  'status': 'Entregue'},
 {'id_pedido': 27,
  'cliente': {'id_cliente': 127, 'nome_cliente': 'Amanda Melo'},
  'compra': {'id_produto': 227,
   'nome_produto': 'Colar de Prata',
   'categoria': 'Acessórios',
   'quantidade': 1,
   'preco_unitario': 120.0,
   'preco_total': 120.0},
  'data_pedido': '2025-01-27',
  'status': 'Pendente'},
 {'id_pedido': 28,
  'cliente': {'id_cliente': 128, 'nome_cliente': 'Bruna Santos'},
  'compra': {'id_produto': 228,
   'nome_produto': 'Fogão 4 Bocas',
   'categoria': 'Eletrodomésticos',
   'quantidade': 1,
   'preco_unitario': 1200.0,
   'preco_total': 1200.0},
  'data_pedido': '2025-01-28',
  'status': 'Entregue'},
 {'id_pedido': 29,
  'cliente': {'id_cliente': 129,

In [18]:
df=pd.json_normalize(dados, sep="_")
df

,id_pedido,data_pedido,status,cliente_id_cliente,cliente_nome_cliente,compra_id_produto,compra_nome_produto,compra_categoria,compra_quantidade,compra_preco_unitario,compra_preco_total
0,26,2025-01-26,Entregue,126,Zeca Costa,226,Cadeira de Escritório,Móveis,1,550.0,550.0
1,27,2025-01-27,Pendente,127,Amanda Melo,227,Colar de Prata,Acessórios,1,120.0,120.0
2,28,2025-01-28,Entregue,128,Bruna Santos,228,Fogão 4 Bocas,Eletrodomésticos,1,1200.0,1200.0
3,29,2025-01-29,Cancelado,129,Carlos Silva,229,Geladeira,Eletrodomésticos,1,3500.0,3500.0
4,30,2025-01-30,Entregue,130,Débora Lima,230,Teclado Mecânico,Eletrônicos,1,300.0,300.0


In [19]:
with open("/content/questao_6.json","r") as e:
  dados=json.load(e)

In [25]:
df=pd.json_normalize(dados,record_path="lista_lojas")
df

,id_loja,nome_loja,cidade,estado,categoria,vendas_mensais,faturamento_mensal
0,5,Loja Épsilon,Salvador,BA,Livros,300,75000.0
1,6,Loja Zeta,Fortaleza,CE,Eletrônicos,100,50000.0
2,7,Loja Eta,Porto Alegre,RS,Roupas,250,87500.0
3,8,Loja Theta,Manaus,AM,Calçados,70,21000.0


In [26]:
with open("/content/questao_7.json") as e:
  dados=json.load(e)

In [28]:
pd.json_normalize(dados)

,lojas_sudeste,lojas_sul
0,"[{'id_loja': 1, 'nome_loja': 'Loja Alpha', 'ci...","[{'id_loja': 3, 'nome_loja': 'Loja Gamma', 'ci..."


In [33]:
df_sudeste=pd.json_normalize(dados, record_path="lojas_sudeste")
df_sudeste

,id_loja,nome_loja,cidade,estado,categoria,vendas_mensais,faturamento_mensal
0,1,Loja Alpha,São Paulo,SP,Eletrônicos,150,45000.0
1,2,Loja Beta,Rio de Janeiro,RJ,Roupas,200,60000.0


In [34]:
df_sul=pd.json_normalize(dados, record_path="lojas_sul")
df_sul

,id_loja,nome_loja,cidade,estado,categoria,vendas_mensais,faturamento_mensal
0,3,Loja Gamma,Belo Horizonte,MG,Calçados,120,36000.0
1,4,Loja Delta,Curitiba,PR,Acessórios,80,24000.0


In [35]:
df=pd.concat([df_sudeste,df_sul])
df

,id_loja,nome_loja,cidade,estado,categoria,vendas_mensais,faturamento_mensal
0,1,Loja Alpha,São Paulo,SP,Eletrônicos,150,45000.0
1,2,Loja Beta,Rio de Janeiro,RJ,Roupas,200,60000.0
0,3,Loja Gamma,Belo Horizonte,MG,Calçados,120,36000.0
1,4,Loja Delta,Curitiba,PR,Acessórios,80,24000.0


In [41]:
with open("/content/questao_8.json","r") as e:
  dados=json.load(e)

In [42]:
pd.json_normalize(dados)

,lojas.lojas_nordeste,lojas.lojas_norte
0,"[{'id_loja': 9, 'nome_loja': 'Loja Iota', 'cid...","[{'id_loja': 11, 'nome_loja': 'Loja Lambda', '..."


In [44]:
df_nordeste=pd.json_normalize(dados,record_path=["lojas","lojas_nordeste"])
df_nordeste

,id_loja,nome_loja,cidade,estado,categoria,vendas_mensais,faturamento_mensal
0,9,Loja Iota,Recife,PE,Acessórios,95,28500.0
1,10,Loja Kappa,Fortaleza,CE,Livros,180,54000.0


In [45]:
df_norte=pd.json_normalize(dados,record_path=["lojas","lojas_norte"])
df_norte

,id_loja,nome_loja,cidade,estado,categoria,vendas_mensais,faturamento_mensal
0,11,Loja Lambda,Manaus,AM,Eletrônicos,220,110000.0
1,12,Loja Mu,Belém,PA,Roupas,160,48000.0


In [52]:
df=pd.concat([df_nordeste,df_norte],ignore_index=True)
df

,id_loja,nome_loja,cidade,estado,categoria,vendas_mensais,faturamento_mensal
0,9,Loja Iota,Recife,PE,Acessórios,95,28500.0
1,10,Loja Kappa,Fortaleza,CE,Livros,180,54000.0
2,11,Loja Lambda,Manaus,AM,Eletrônicos,220,110000.0
3,12,Loja Mu,Belém,PA,Roupas,160,48000.0
